
Enforce data quality
The following operations are performed in silver tables:
1) Delete Duplicate / Data deduplication
2) Column Schema evolution / Schema evolution
3) DataTypes Handling / Type casting
4) Null Records Handling / Handling of null and missing values
5) Trim / Data quality checks and enforcement
6) Case Conversion
7) Maintain Order of Data
8) Joins

In [0]:
%sql
INSERT INTO learning_catalog.learning_silver.CleanedMonthlySales
SELECT DISTINCT
    CAST(TRIM(sale_id)AS INTEGER) as sale_id, 
    UPPER(TRIM(product)) as product,
    UPPER(TRIM(category)) as category,
    CAST(TRIM(quantity)AS INTEGER) as quantity ,
    CAST(TRIM(price)AS DOUBLE) as price,
    TO_DATE(TRIM(sale_date),'dd-mm-yyyy') as sale_date,  --> 28-01-2024
    UPPER(TRIM(region)) as region,
    current_timestamp() as ingest_ts
   from learning_catalog.learning_bronze.rawMonthlySales
   where ingest_ts > (
                        select coalesce(max(ingest_ts),TO_DATE('1990-01-01','yyyy-MM-dd')) 
                        from learning_catalog.learning_silver.CleanedMonthlySales)
                        ORDER BY TRY_CAST(TRIM(sale_id)AS INTEGER);
    

In [0]:
%sql
INSERT INTO learning_catalog.learning_silver.CleanedMonthlySales
SELECT DISTINCT
    CAST(TRIM(sale_id) AS INTEGER) AS sale_id,
    UPPER(TRIM(product)) AS product,
    UPPER(TRIM(category)) AS category,
    COALESCE(CAST(TRIM(quantity) AS INTEGER), 0) AS quantity,   -- null value replacing with 0
    COALESCE(CAST(TRIM(price) AS DOUBLE), 0.0) AS price,        -- null value replacing with 0
    COALESCE(TO_DATE(TRIM(sale_date), 'yyyy-MM-dd'), DATE '1900-01-01') AS sale_date,
    COALESCE(UPPER(TRIM(region)), 'UNSPECIFIED') AS region,      -- null value replacing with unspecified
    CURRENT_TIMESTAMP AS ingest_ts
FROM learning_catalog.learning_bronze.rawMonthlySales b
WHERE b.ingest_ts > (
    SELECT COALESCE(MAX(s.ingest_ts), TIMESTAMP '1990-01-01 00:00:00')
    FROM learning_catalog.learning_silver.CleanedMonthlySales s
);
